# 2ª Submissão: Extração de Features + LSTM

Este notebook contém o pipeline completo para a 2ª submissão: carregar os dados (incluindo datasets externos), aplicar a extração de features TF-IDF e Handcrafted, treinar o modelo 
LSTM e, por fim, processar e guardar as previsões do conjunto de teste num CSV.

In [18]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from datasets import load_dataset


## 1. Carregar Datasets

In [19]:
def get_prof_dataset(n_lines: int = 10000) -> pd.DataFrame:
    try:
        dataset_path = "../data/dataset-exemplos.csv"
        df = pd.read_csv(dataset_path, sep=";")
        df["id"] = df["ID"]
    except:
        dataset_path = "../data/dataset_treino.csv"
        df = pd.read_csv(dataset_path)
        df["id"] = df.index
        df["Label"] = df["source_name"]

    n_lines = min(n_lines, len(df))
    return df[["id", "Text", "Label"]].sample(n_lines, random_state=42).reset_index(drop=True)

def get_otb_dataset(n_lines: int = 10000) -> pd.DataFrame:
    try:
        dataset = load_dataset("MLNTeam-Unical/OpenTuringBench", name="in_domain")
        df_train = dataset["train"].to_pandas()
        df_test = dataset["test"].to_pandas()
        df = pd.concat([df_train, df_test], ignore_index=True)

        mapping_classes = {
            "meta-llama": "Meta",
            "qwen": "OpenAI",
            "google": "Google",
            "anthropic": "Anthropic",
        }
        df["id"] = df["url"]
        df["Text"] = df["content"]
        df["Label"] = df["model"].apply(lambda x: mapping_classes.get(x.split("/")[0].lower(), "Others"))
        df = df[df["Label"] != "Others"]
        n_lines = min(n_lines, len(df))
        return df[["id", "Text", "Label"]].sample(n_lines, random_state=42).reset_index(drop=True)
    except:
        return pd.DataFrame(columns=["id", "Text", "Label"])

def get_atdp_dataset(n_lines: int = 10000) -> pd.DataFrame:
    try:
        dataset = load_dataset("artem9k/ai-text-detection-pile")
        df = dataset["train"].to_pandas()
        df["Text"] = df["text"]
        df = df[df["source"] == "human"]
        df["Label"] = df["source"].apply(lambda x: "Human" if x == "human" else "")
        n_lines = min(n_lines, len(df))
        return df[["id", "Text", "Label"]].sample(n_lines, random_state=42).reset_index(drop=True)
    except:
        return pd.DataFrame(columns=["id", "Text", "Label"])

def get_ap_dataset(n_lines: int = 3000) -> pd.DataFrame:
    try:
        dataset = load_dataset("Anthropic/persuasion")
        df = dataset["train"].to_pandas()
        df["id"] = df["worker_id"]
        df["Text"] = df["argument"]
        df["Label"] = df["source"].apply(lambda x: "Anthropic" if x.startswith("Claude") else "Human")
        n_lines = min(n_lines, len(df))
        return df[["id", "Text", "Label"]].sample(n_lines, random_state=42).reset_index(drop=True)
    except:
        return pd.DataFrame(columns=["id", "Text", "Label"])

def get_datasets() -> pd.DataFrame:
    df_prof = get_prof_dataset()
    df_otb = get_otb_dataset()
    df_atdp = get_atdp_dataset()
    df_ap = get_ap_dataset()

    df = pd.concat([df_prof, df_otb, df_atdp, df_ap], ignore_index=True)
    return df

df_full = get_datasets()
print(df_full["Label"].value_counts())


Label
Human        10431
human         4876
Anthropic     3733
Google        3388
OpenAI        3328
Meta          3284
openai        1556
google        1212
meta          1192
Name: count, dtype: int64


## 2. Funções de Extração de Features

In [20]:
import string
import re
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

stop_words = {
    "a", "an", "the", "and", "or", "but", "if", "while",
    "with", "without", "in", "on", "at", "by", "for", "to", "from",
}

# 1. Limpeza de Texto
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_text_clean(text):
    text = str(text).lower()
    text = "".join(char for char in text if char.isalnum() or char.isspace())
    return text

# 2. Codificação das Labels
def encode_labels(labels):
    unique_labels = sorted(list(set(labels)))
    mapping = {label: idx for idx, label in enumerate(unique_labels)}
    y = np.array([mapping[l] for l in labels])
    return y, mapping

# 3. Inicialização do TF-IDF
def build_vectorizer():
    return TfidfVectorizer(max_features=3000, stop_words='english')

# 4. Criação das Handcrafted Features (33 Features Avancadas!)
def extract_features(raw_text, clean_text):
    words = clean_text.split()
    num_words = len(words)
    raw_lower = str(raw_text).lower()

    sentence_split = str(raw_text).replace("!", ".").replace("?", ".").split(".")
    sentences = [s.strip() for s in sentence_split if s.strip()]
    sentence_lengths = [len(preprocess_text_clean(s).split()) for s in sentences]

    word_counts = Counter(words)
    punctuation = {p: str(raw_text).count(p) for p in [",", ".", ";", ":"]}
    n_chars = len(str(raw_text)) if raw_text else 1

    features = {}
    features["length_chars"] = len(str(raw_text))
    features["num_words"] = num_words
    features["num_sentences"] = len(sentences)
    features["avg_word_length"] = np.mean([len(w) for w in words]) if num_words > 0 else 0.0
    features["avg_sentence_length"] = np.mean(sentence_lengths) if sentence_lengths else 0.0
    features["sentence_length_std"] = np.std(sentence_lengths) if len(sentence_lengths) > 1 else 0.0
    features["type_token_ratio"] = len(set(words)) / num_words if num_words > 0 else 0.0
    features["hapax_ratio"] = sum(1 for c in word_counts.values() if c == 1) / num_words if num_words > 0 else 0.0
    features["repetition_ratio"] = max(word_counts.values()) / num_words if num_words > 0 else 0.0

    for p, count in punctuation.items():
        key = f"punct_{p}_ratio".replace(".", "dot")
        features[key] = count / n_chars

    original_tokens = [token for token in preprocess_text_clean(raw_lower).split() if token]
    features["stopword_ratio"] = (sum(1 for token in original_tokens if token in stop_words) / len(original_tokens) if original_tokens else 0.0)
    features["uppercase_ratio"] = sum(1 for ch in str(raw_text) if ch.isupper()) / n_chars
    features["digit_ratio"] = sum(1 for ch in str(raw_text) if ch.isdigit()) / n_chars
    features["whitespace_ratio"] = sum(1 for ch in str(raw_text) if ch.isspace()) / n_chars

    n_contractions = len(re.findall(r"\b\w+'\w+\b", str(raw_text)))
    features["contraction_ratio"] = n_contractions / num_words if num_words > 0 else 0.0

    all_caps_words = [w for w in str(raw_text).split() if len(w) > 1 and w.isupper()]
    features["allcaps_word_ratio"] = len(all_caps_words) / num_words if num_words > 0 else 0.0
    features["missing_space_after_punct"] = len(re.findall(r"[.!?,;:][A-Za-z]", str(raw_text))) / n_chars
    features["fragment_ratio"] = (sum(1 for sl in sentence_lengths if sl <= 3) / len(sentence_lengths) if sentence_lengths else 0.0)
    features["runon_ratio"] = (sum(1 for sl in sentence_lengths if sl >= 40) / len(sentence_lengths) if sentence_lengths else 0.0)

    raw_word_tokens = re.findall(r"\b\w+\b", raw_lower)
    filler_words = {"tbh", "lol", "omg", "idk", "imo", "btw", "ngl", "smh", "fyi", "like", "just", "really"}
    features["filler_word_ratio"] = (sum(1 for w in raw_word_tokens if w in filler_words) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    ai_connectors = {"furthermore", "moreover", "additionally", "consequently", "nevertheless", "therefore", "thus"}
    features["ai_connector_ratio"] = (sum(1 for w in raw_word_tokens if w in ai_connectors) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    first_person = {"i", "me", "my", "mine", "myself", "we", "us", "our", "ours"}
    features["first_person_ratio"] = (sum(1 for w in raw_word_tokens if w in first_person) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    features["question_density"] = str(raw_text).count("?") / len(sentences) if sentences else 0.0

    if len(words) > 1:
        bigrams = list(zip(words[:-1], words[1:]))
        features["bigram_uniqueness"] = len(set(bigrams)) / len(bigrams)
    else:
        features["bigram_uniqueness"] = 0.0

    adverbs = {"very", "quite", "extremely", "remarkably", "clearly"}
    features["adverb_ratio"] = (sum(1 for w in raw_word_tokens if w in adverbs) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    modals = {"would", "could", "should", "might", "may", "must", "can", "will", "shall"}
    features["modal_ratio"] = (sum(1 for w in raw_word_tokens if w in modals) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    discourse_connectors = {"however", "therefore", "furthermore", "moreover", "consequently"}
    features["transition_ratio"] = (sum(1 for w in raw_word_tokens if w in discourse_connectors) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    be_verbs = {"is", "was", "are", "were", "be", "been", "being"}
    passive_constructions = sum(1 for w in raw_word_tokens if w in be_verbs)
    features["passive_voice_ratio"] = passive_constructions / len(sentences) if sentences else 0.0

    citation_patterns = len(re.findall(r"\b\w+,\s*\d{4}\b|\bet\s+al\b", str(raw_text)))
    features["citation_density"] = citation_patterns / len(sentences) if sentences else 0.0

    hedging_words = {"suggest", "appears", "likely", "unlikely", "possibly", "perhaps"}
    features["hedging_ratio"] = (sum(1 for w in raw_word_tokens if w in hedging_words) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    return features

def build_handcrafted_matrix(raw_texts, clean_texts):
    feature_dicts = [extract_features(r, c) for r, c in zip(raw_texts, clean_texts)]
    feature_names = sorted(feature_dicts[0].keys())
    matrix = np.array([[fd[name] for name in feature_names] for fd in feature_dicts], dtype=float)
    return matrix, feature_names

# 5. Normalização das Handcrafted Features (Z-Score)
def standardize_train_test(train_mat, val_mat):
    mean = np.mean(train_mat, axis=0)
    std = np.std(train_mat, axis=0) + 1e-8 
    train_std = (train_mat - mean) / std
    val_std = (val_mat - mean) / std
    return train_std, val_std, mean, std

## 3. Preparação do Pipeline e Treino

In [21]:
# Preparar divisão de Treino/Validação
df_full = df_full.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df_full) * 0.8)

train_texts = df_full["Text"][:split_idx].fillna("").astype(str).tolist()
train_labels = df_full["Label"][:split_idx].tolist()

val_texts = df_full["Text"][split_idx:].fillna("").astype(str).tolist()
val_labels = df_full["Label"][split_idx:].tolist()

y_train, mapping = encode_labels(train_labels)
y_val, _ = encode_labels(val_labels)

# Iniciar TF-IDF
vectorizer = build_vectorizer()
train_clean1 = [preprocess_text(t) for t in train_texts]
val_clean1 = [preprocess_text(t) for t in val_texts]

X_train_tfidf = vectorizer.fit_transform(train_clean1).toarray()
X_val_tfidf = vectorizer.transform(val_clean1).toarray()

# Handcrafted Features
train_clean2 = [preprocess_text_clean(t) for t in train_texts]
val_clean2 = [preprocess_text_clean(t) for t in val_texts]

X_train_hc, fnames = build_handcrafted_matrix(train_texts, train_clean2)
X_val_hc, _ = build_handcrafted_matrix(val_texts, val_clean2)

X_train_hc_std, X_val_hc_std, hc_mean, hc_std = standardize_train_test(X_train_hc, X_val_hc)

# Concatenar as matrizes
X_train_full = np.hstack([X_train_tfidf, X_train_hc_std])
X_val_full = np.hstack([X_val_tfidf, X_val_hc_std])

print("Shape do treino:", X_train_full.shape)


Shape do treino: (26400, 3033)


## 4. LSTM Dataset e Classificador

In [22]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        if hasattr(X, "toarray"):
            X = X.toarray()
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, embed_dim=128, hidden_dim=128, n_classes=6):
        super().__init__()
        self.embedding = nn.Linear(input_dim, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.unsqueeze(1)
        output, (h, c) = self.lstm(x)
        return self.fc(h[-1])

train_dataset = TextDataset(X_train_full, y_train)
val_dataset = TextDataset(X_val_full, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMClassifier(input_dim=X_train_full.shape[1], n_classes=len(mapping)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X_batch.size(0)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
        
    acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/total:.4f} | Acc: {acc:.4f}")


Epoch 1/10 | Loss: 0.6568 | Acc: 0.7752
Epoch 2/10 | Loss: 0.1966 | Acc: 0.9344
Epoch 3/10 | Loss: 0.1161 | Acc: 0.9630
Epoch 4/10 | Loss: 0.0731 | Acc: 0.9778
Epoch 5/10 | Loss: 0.0505 | Acc: 0.9848
Epoch 6/10 | Loss: 0.0346 | Acc: 0.9908
Epoch 7/10 | Loss: 0.0249 | Acc: 0.9934
Epoch 8/10 | Loss: 0.0188 | Acc: 0.9953
Epoch 9/10 | Loss: 0.0150 | Acc: 0.9961
Epoch 10/10 | Loss: 0.0120 | Acc: 0.9970


## Gravacao do Modelo e Variáveis

In [24]:
import pickle
import os
os.makedirs('../models/subm2_model', exist_ok=True)

torch.save(model.state_dict(), '../models/subm2_model/lstm_model.pt')
with open('../models/subm2_model/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
with open('../models/subm2_model/hc_stats.pkl', 'wb') as f:
    pickle.dump({'mean': hc_mean, 'std': hc_std}, f)
with open('../models/subm2_model/mapping.pkl', 'wb') as f:
    pickle.dump(mapping, f)
print("Modelo e artefatos guardados com sucesso em ../models/subm2_model/")


Modelo e artefatos guardados com sucesso em ../models/subm2_model/


## 5. Correr no Dataset de Teste e Guardar o CSV Final

In [25]:
test_df = pd.read_csv('../data/dataset_teste.csv')

def format_submit(test_df, vectorizer, hc_mean, hc_std, mapping):
    test_texts = test_df['Text'].fillna("").astype(str).tolist()
    
    test_clean1 = [preprocess_text(t) for t in test_texts]
    X_test_tfidf = vectorizer.transform(test_clean1).toarray()
    
    test_clean2 = [preprocess_text_clean(t) for t in test_texts]
    X_test_hc, _ = build_handcrafted_matrix(test_texts, test_clean2)
    X_test_hc_std = (X_test_hc - hc_mean) / hc_std
    
    X_test_full = np.hstack([X_test_tfidf, X_test_hc_std])

    test_dataset = TextDataset(X_test_full, np.zeros(len(X_test_full)))
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    model.eval()
    all_preds = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
    
    submission_df = test_df.copy()
    submission_df['predicted_class'] = all_preds
    
    reverse_mapping = {v: k for k, v in mapping.items()}
    submission_df['predicted_source'] = submission_df['predicted_class'].map(reverse_mapping)
    
    return submission_df

submission_df = format_submit(test_df, vectorizer, hc_mean, hc_std, mapping)

csv_out = '../subm1/subm2-MIA-B.csv'
submission_df.to_csv(csv_out, index=False)
print(f"Predições criadas no ficheiro {csv_out}")
submission_df[['Text', 'predicted_source']].head()


Predições criadas no ficheiro ../subm1/subm2-MIA-B.csv


,Text,predicted_source
0,Heliora Logistics entered voluntary administra...,Anthropic
1,Brendan Rodgers has admitted to watching Liver...,human
2,I show that several observable properties of b...,human
3,The analysis considered the correlation betwee...,openai
4,Riboswitch aptamer-expression platform communi...,Anthropic
